# CerebroSpheroid: Google Colab training

Trains the non-autoregressive volumetric-manifold model (146,618,503 parameters) on the seeded cognitive curriculum with the zero-feedback objective from `src/spheroid_v3.py`.

This notebook uses `colab/colab_cognitive.py`, which selects the best checkpoint on a frozen held-out set and writes provenance beside every weights file. A Colab T4 (16 GB) runs it directly; the laptop RTX 3050 cannot hold the unrolled activations, which is the point of running here.

In [ ]:
# 1. Confirm the GPU runtime and install dependencies.
# Runtime -> Change runtime type -> T4 GPU (or A100 on Colab Pro) before running this.
!nvidia-smi
!pip install -q torch transformers datasets pyarrow

In [ ]:
# 2. Optional: mount Google Drive so checkpoints survive a Colab disconnect.
from google.colab import drive
try:
    drive.mount('/content/drive')
    print('Drive mounted; checkpoints will mirror to it.')
except Exception as e:
    print('Drive not mounted (continuing with local-only checkpoints):', e)

In [ ]:
# 3. Unpack the self-contained project (code + cerebro_cognitive dataset + 1m checkpoints).
# Upload the new zip into this notebook's files panel, then run this cell.
import os, glob, zipfile
cands = ['/content/reasoning_agent_colab_selfcontained.zip'] + sorted(glob.glob('/content/*colab*.zip'))
z = next((p for p in cands if os.path.exists(p)), None)
if z is None:
    raise SystemExit('No Colab zip found in /content. Upload it via the Files panel first.')
print('Unzipping', z, '(', round(os.path.getsize(z)/1e6, 1), 'MB )')
with zipfile.ZipFile(z) as zf:
    zf.extractall('/content')
os.chdir('/content/reasoning_agent')
!ls data/cerebro_cognitive && ls checkpoints

In [ ]:
# 4. Launch cognitive training on the bundled train split, warm-started from the shipped weights.
# batch_size 4 is the memory-bound micro-batch that fits a T4 across the ~80-step unroll;
# grad_accum 4 gives effective batch 16. Selection is on the frozen val split, not train loss.
!python3 colab/colab_cognitive.py \
    --scale laptop \
    --steps 2000 \
    --batch_size 4 \
    --grad_accum 4 \
    --lr 3e-4 \
    --eval_interval 50 \
    --drive_dir /content/drive/MyDrive/cerebro_checkpoints \
    --resume checkpoints/cerebro_spheroid_1m_best.pt

## Interactive reasoning check

Load the best checkpoint this session produced (falling back to the shipped `1m_best`) and run the zero-feedback deliberate-then-emit loop on a prompt.

In [ ]:
import os
from src.inference import ReasoningAgent
ckpt = 'checkpoints/cerebro_spheroid_colab_best.pt'
if not os.path.exists(ckpt):
    ckpt = 'checkpoints/cerebro_spheroid_1m_best.pt'
print('Loading', ckpt)
agent = ReasoningAgent(checkpoint_path=ckpt, arch='auto')
q = 'If 4 books cost $20, how much do 6 books cost?'
print('Q:', q)
print('A:', agent.generate(q, thinking_cycles=4, max_tokens=48, temperature=0.0))